In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)
from datetime import datetime


In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- mosmix_access_dataframe ---
FIX_MOSMIX_ACCESS_DATAFRAME_DATA_DICT = {"key": [1, 2, 3]}

# --- mosmix_access_datetime ---
FIX_MOSMIX_ACCESS_DATETIME_TIMESTEPS = type("TS", (), {"getchildren": lambda self: [type("E", (), {"text": "2020-01-01T12:00:00"})()] * 3})()

print("✅ Fixtures loaded")

In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_mosmix_access_dataframe(data_dict):
    yield pd.DataFrame.from_dict(data_dict)
    return None

def before_mosmix_access_datetime(timesteps):
    return pd.DatetimeIndex([pd.Timestamp(i.text) for i in timesteps.getchildren()])
    return None

In [ ]:
# ── Generated wrappers (experiment-generated Polars) ─────────────────────────

def gen_mosmix_access_dataframe(data_dict):
    pd = pl  # LLM used `import polars as pd`
    yield pd.DataFrame(data_dict)
    return None

def gen_mosmix_access_datetime(timesteps):

    pl.Series([pd.Timestamp(i.text) for i in timesteps.getchildren()])
    return None

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: mosmix_access_datetime ===

# L1 smoke – generated
try:
    _r = gen_mosmix_access_datetime(FIX_MOSMIX_ACCESS_DATETIME_TIMESTEPS)
    print("✅ L1 smoke gen_mosmix_access_datetime: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_mosmix_access_datetime: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_mosmix_access_datetime(FIX_MOSMIX_ACCESS_DATETIME_TIMESTEPS)
    print("✅ L1 smoke before_mosmix_access_datetime: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_mosmix_access_datetime: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence – compare datetime sequence values.
try:
    _rb = [pd.Timestamp(x) for x in before_mosmix_access_datetime(FIX_MOSMIX_ACCESS_DATETIME_TIMESTEPS)]
    _rg = [pd.Timestamp(x) for x in gen_mosmix_access_datetime(FIX_MOSMIX_ACCESS_DATETIME_TIMESTEPS).to_list()]
    if _rb == _rg:
        print("✅ L2 equivalence mosmix_access_datetime sequence: MATCH")
    else:
        print(f"❌ L2 equivalence mosmix_access_datetime sequence: MISMATCH — before={_rb}, gen={_rg}")
except Exception as _e:
    print(f"❌ L2 equivalence mosmix_access_datetime: setup error — {type(_e).__name__}: {_e}")

# L3 edge - an empty XML timestep collection returns an empty datetime sequence.
try:
    _empty_timesteps = SimpleNamespace(getchildren=lambda: [])
    _rb = [pd.Timestamp(x) for x in before_mosmix_access_datetime(_empty_timesteps)]
    _rg = [pd.Timestamp(x) for x in gen_mosmix_access_datetime(_empty_timesteps).to_list()]
    if _rb == _rg == []:
        print("✅ L3 edge mosmix_access_datetime empty timesteps: MATCH")
    else:
        print(f"❌ L3 edge mosmix_access_datetime empty timesteps: MISMATCH - before={_rb}, gen={_rg}")
except Exception as _e:
    print(f"❌ L3 edge mosmix_access_datetime empty timesteps: {type(_e).__name__}: {_e}")
